# 10.02 -- EAGLE: Extrapolation Algorithm for Greater Language-model Efficiency

**Module:** 10 -- Advanced Inference  
**Notebook:** 2 of 5  
**Prerequisites:** 10.01 (Speculative Decoding)

---

## Learning Objectives

1. Understand why standard speculative decoding is limited by the draft model alignment gap
2. Derive the EAGLE architecture: using the target model own hidden states to drive drafting
3. Implement the EAGLE feature-level drafter from scratch
4. Measure acceptance rate and speedup vs independent small-model speculation
5. Understand EAGLE-2 dynamic draft length based on drafter confidence

---

## 1. The Draft Model Alignment Gap

Standard speculative decoding trains a separate small model (e.g. 68 M parameters) to draft
tokens for a large target model (e.g. 7 B parameters). The acceptance rate -- and therefore
the throughput speedup -- is bounded by how well the small model predicts what the large model
would have generated.

The fundamental problem: a 68 M model does not reason the same way as a 7 B model.
No matter how well the small model is trained, there is an alignment gap between their
distributions that caps the acceptance rate, typically at 65-78 percent per token.

**EAGLE insight**: instead of building a separate draft model from scratch, use the target
model own internal representations. The hidden state at the last layer already encodes
everything the target model knows about the context. A lightweight head that predicts the
next hidden state -- and therefore the next token -- should align far better with the target
than any separately trained model can.

---

## 2. EAGLE Architecture

```
Standard Speculative Decoding:
  Draft model (separate, small):  tokens --> [small transformer] --> draft token probs
  Target model (large):           tokens --> [large transformer] --> accept / reject

EAGLE Architecture:
  Target prefill:   tokens --> [large transformer] --> feature vector f_t
  EAGLE drafter:    (f_t, embed_t) --> [1 decoder layer] --> f_{t+1} --> token probs
  Target verify:    tokens --> [large transformer] --> accept / reject

Key: the drafter predicts the next hidden state, not tokens directly.
     This keeps drafting in the target model representation space.
```

The EAGLE drafter is a single transformer decoder layer with a linear fusion input that
combines the last feature vector with the current token embedding. It is roughly 2-4 percent
of the target model parameter count but achieves 85-92 percent token acceptance when trained.


In [ ]:
# Environment setup.
#
# EAGLE requires access to the target model hidden states at every layer.
# We access these via output_hidden_states=True in the HuggingFace forward call.
# The EAGLE drafter is implemented from scratch using standard PyTorch layers.
# We demonstrate on GPT-2 (117 M) to keep compute requirements CPU-friendly.

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
import time
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')


## 3. Implementing the EAGLE Drafter From Scratch


In [ ]:
# EAGLE drafter architecture.
#
# The drafter has three components:
#
#   1. Linear fusion layer: concatenates the previous hidden state (f_{t-1})
#      with the current token embedding (e_t) and projects back to hidden_size.
#      Dimension: (hidden_size * 2) --> hidden_size.
#      This is the context-injection step: it gives the drafter both the semantic
#      content from the feature vector and the surface-form identity of the token.
#
#   2. Single transformer decoder layer: one round of multi-head self-attention
#      followed by a feed-forward network. This is all the 'reasoning' the drafter
#      does. Empirically, one layer captures the local token transition dynamics;
#      additional layers give marginal gain at significant compute cost.
#
#   3. LM head: linear projection from hidden_size to vocab_size.
#      In production, this is weight-tied to the target model embedding table
#      so the output distribution lives in the same probability space.
#
# Parameter count for a GPT-2 sized target (hidden=768, vocab=50257, heads=12):
#   Fusion layer:  768 * 2 * 768        =   1.2 M
#   Attn + FFN:    ~7 M
#   LM head:       768 * 50257          =  38.6 M
#   Total:        ~47 M  vs  117 M target  (40 percent of target size)
#
# For a 7 B target (hidden=4096, vocab=32000):
#   Total drafter: ~150 M  vs  7000 M  (about 2 percent)

class EAGLEDrafter(nn.Module):
    """
    Single-layer EAGLE feature drafter.

    Takes the target model last hidden state and the current token embedding,
    fuses them, runs one transformer decoder layer, and returns predicted
    next-token logits and the predicted next hidden state.
    """

    def __init__(self, hidden_size: int, num_heads: int, vocab_size: int,
                 ffn_mult: int = 4):
        super().__init__()
        self.hidden_size = hidden_size

        # Fusion: concat [feature_vec, token_embed] --> hidden_size
        self.fc_fusion = nn.Linear(hidden_size * 2, hidden_size, bias=False)

        # Transformer decoder layer
        self.ln1  = nn.LayerNorm(hidden_size)
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=num_heads,
            dropout=0.0, batch_first=True,
        )
        self.ln2 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * ffn_mult),
            nn.GELU(),
            nn.Linear(hidden_size * ffn_mult, hidden_size),
        )

        # LM head (weight-tied to target embedding in production)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)

    def forward(
        self,
        feature_vec: torch.Tensor,            # (B, hidden_size)
        token_embed: torch.Tensor,            # (B, hidden_size)
        past_features: Optional[torch.Tensor] = None,  # (B, t, hidden_size)
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        One EAGLE draft step.

        Returns:
            draft_logits:    (B, vocab_size) -- next-token prediction
            output_feature:  (B, hidden_size) -- predicted next hidden state;
                              feed this back as feature_vec in the next step
        """
        # Fuse feature vector and token embedding
        fused = self.fc_fusion(
            torch.cat([feature_vec, token_embed], dim=-1)
        ).unsqueeze(1)  # (B, 1, hidden_size)

        # Build attention context from accumulated past features + current
        context = fused if past_features is None else torch.cat([past_features, fused], dim=1)

        # Self-attention block
        x          = self.ln1(context)
        attn_out, _ = self.attn(x, x, x)
        context    = context + attn_out

        # FFN block
        context = context + self.ffn(self.ln2(context))

        # Extract last position
        output_feature = context[:, -1, :]   # (B, hidden_size)
        draft_logits   = self.lm_head(output_feature)
        return draft_logits, output_feature


# Instantiate for GPT-2 small: hidden=768, heads=12, vocab=50257
drafter = EAGLEDrafter(hidden_size=768, num_heads=12, vocab_size=50257).to(device)

drafter_params = sum(p.numel() for p in drafter.parameters())
target_params  = 117_000_000   # GPT-2 small

print('EAGLE drafter parameter count:')
for name, mod in drafter.named_children():
    n = sum(p.numel() for p in mod.parameters())
    print(f'  {name:<15} {n:>10,}')
print(f'  {"Total":<15} {drafter_params:>10,}')
print(f'  Target model   {target_params:>10,}')
print(f'  Ratio:         {drafter_params / target_params:.1%}')


In [ ]:
# EAGLE generation loop.
#
# The four-phase loop:
#
#   Phase 1 -- Prefill (once per request):
#     Run the full target model on the prompt.
#     Collect the last hidden state f_0 and the first-token logits.
#
#   Phase 2 -- EAGLE Draft:
#     For each draft step i = 1 .. gamma:
#       a) Look up the token embedding for draft token_{i-1}.
#       b) Run the EAGLE drafter: (f_{i-1}, embed_{i-1}) --> logits_i, f_i.
#       c) Sample: token_i ~ softmax(logits_i).
#     Cost: gamma * (one linear layer + one attention layer) -- very cheap.
#
#   Phase 3 -- Target Verify:
#     Run the target model on (prompt + all draft tokens) in one forward pass.
#     The KV cache covers the prompt; we only compute attention over draft tokens.
#     This produces target probabilities at each draft position.
#
#   Phase 4 -- Speculative Acceptance:
#     Accept token i with probability min(1, p_target_i / p_draft_i).
#     Stop at first rejection; resample from the corrected distribution.
#     On average, gamma * alpha tokens are accepted per target model call.

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

target = AutoModelForCausalLM.from_pretrained(
    'gpt2', output_hidden_states=True
).to(device)
target.eval()

print(f'Target model: {sum(p.numel() for p in target.parameters()):,} parameters')


def eagle_prefill(ids: torch.Tensor):
    """Run target model on prompt; return last-token logits and last hidden state."""
    with torch.no_grad():
        out = target(ids, output_hidden_states=True)
    # out.hidden_states: tuple of (n_layers+1) tensors each (B, seq, hidden)
    last_hidden = out.hidden_states[-1][:, -1, :]   # (B, hidden_size)
    logits      = out.logits[:, -1, :]               # (B, vocab_size)
    return logits, last_hidden


def eagle_draft(feature_vec, start_token, gamma=4, temperature=1.0):
    """
    Run gamma EAGLE draft steps starting from start_token.

    Returns draft token ids, their probability distributions,
    and the accumulated feature vectors.
    """
    token          = start_token.clone()
    feat           = feature_vec.clone()
    draft_tokens   = []
    draft_probs    = []
    past           = None

    for _ in range(gamma):
        with torch.no_grad():
            tok_embed = target.transformer.wte(token)   # (B, hidden_size)
            logits, feat = drafter(feat, tok_embed, past)

        probs   = F.softmax(logits / max(temperature, 1e-6), dim=-1)
        sampled = torch.multinomial(probs, num_samples=1).squeeze(-1)

        draft_tokens.append(sampled.item())
        draft_probs.append(probs.squeeze(0))

        token = sampled
        past  = (feat.unsqueeze(1) if past is None
                 else torch.cat([past, feat.unsqueeze(1)], dim=1))

    return draft_tokens, draft_probs


def target_verify(prompt_ids, draft_tokens):
    """One target model forward pass over prompt + draft tokens."""
    all_ids = torch.cat([
        prompt_ids,
        torch.tensor(draft_tokens, device=device).unsqueeze(0)
    ], dim=1)
    with torch.no_grad():
        out = target(all_ids)
    prompt_len  = prompt_ids.shape[1]
    target_probs = []
    for i in range(len(draft_tokens)):
        logits = out.logits[0, prompt_len - 1 + i, :]
        target_probs.append(F.softmax(logits, dim=-1))
    return target_probs


def speculative_accept(draft_probs, target_probs, draft_tokens):
    """
    Standard speculative decoding acceptance criterion.

    Accept token i with probability min(1, p_target / p_draft).
    On rejection resample from the corrected distribution (p_target - p_draft).
    Returns accepted token list and count of accepted tokens before first rejection.
    """
    accepted = []
    for tok, p_d, p_t in zip(draft_tokens, draft_probs, target_probs):
        ratio = (p_t[tok] / (p_d[tok] + 1e-10)).item()
        if torch.rand(1).item() < min(1.0, ratio):
            accepted.append(tok)
        else:
            corrected = (p_t - p_d).clamp(min=0)
            s = corrected.sum()
            resampled = (torch.multinomial(corrected / s, 1).item() if s > 0
                         else torch.multinomial(p_t, 1).item())
            accepted.append(resampled)
            return accepted, len(accepted) - 1
    return accepted, len(draft_tokens)


# Quick demo
prompt   = 'The attention mechanism in transformers allows'
ids      = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
GAMMA    = 4

logits_0, feat_0  = eagle_prefill(ids)
first_tok         = torch.multinomial(F.softmax(logits_0, dim=-1), 1).squeeze(-1)
d_tokens, d_probs = eagle_draft(feat_0, first_tok, gamma=GAMMA)
t_probs           = target_verify(ids, [first_tok.item()] + d_tokens)
all_d_probs       = [F.softmax(logits_0, dim=-1).squeeze()] + d_probs
accepted, n_acc   = speculative_accept(all_d_probs, t_probs,
                                       [first_tok.item()] + d_tokens)

print(f'Prompt:        {prompt}')
print(f'Draft tokens:  {tokenizer.convert_ids_to_tokens(d_tokens)}')
print(f'Accepted:      {n_acc} / {GAMMA}')
print(f'Generated:     {tokenizer.decode(accepted)}')


In [ ]:
# Acceptance rate measurement and speedup analysis.
#
# We measure acceptance rate on 10 test prompts for three conditions:
#
#   1. EAGLE with random drafter weights (untrained baseline).
#      This represents the floor: the drafter guesses randomly.
#
#   2. Oracle drafter: draft tokens sampled directly from the target model.
#      This represents the ceiling: acceptance rate = 100 percent.
#
#   3. Published EAGLE results with trained drafter on LLaMA-2 7B.
#      Shown as a reference line from the EAGLE paper.
#
# The gap between random and oracle shows the potential gain from training.
# The theoretical speedup formula is:
#   speedup ~ 1 + gamma * alpha
# where alpha is the mean per-token acceptance rate.
# At alpha=0.90 and gamma=5: speedup ~ 5.5x relative to one token per call.

TEST_PROMPTS = [
    'The transformer architecture uses attention to',
    'Machine learning models are trained by minimising',
    'Natural language processing involves understanding',
    'The gradient descent algorithm updates weights by',
    'Attention mechanisms allow the model to focus on',
    'The loss function measures the difference between',
    'Backpropagation computes gradients using the chain',
    'The softmax function converts raw logits into',
    'Word embeddings represent tokens as dense vectors in',
    'Fine-tuning a pretrained language model requires',
]

GAMMA = 4
random_acc = []
oracle_acc = []

for prompt in TEST_PROMPTS:
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)

    logits_0, feat_0  = eagle_prefill(ids)
    first_tok         = torch.multinomial(F.softmax(logits_0, dim=-1), 1).squeeze(-1)

    # Random drafter acceptance
    d_tokens, d_probs = eagle_draft(feat_0, first_tok, gamma=GAMMA)
    full_d            = [first_tok.item()] + d_tokens
    t_probs           = target_verify(ids, full_d)
    all_d_probs       = [F.softmax(logits_0, dim=-1).squeeze()] + d_probs
    _, n_random       = speculative_accept(all_d_probs, t_probs, full_d)
    random_acc.append(n_random / GAMMA)

    # Oracle: draft = sample from target
    oracle_tokens = [torch.multinomial(p, 1).item() for p in t_probs[:GAMMA]]
    _, n_oracle   = speculative_accept(t_probs[:GAMMA], t_probs[:GAMMA], oracle_tokens)
    oracle_acc.append(n_oracle / GAMMA)


fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# Panel 1: per-prompt acceptance rates
ax = fig.add_subplot(gs[0, 0])
x = range(1, len(TEST_PROMPTS) + 1)
ax.plot(x, random_acc, 'o-', color='#d62728', lw=2, ms=6,
        label=f'EAGLE random drafter  mean={np.mean(random_acc):.2f}')
ax.plot(x, oracle_acc, 's-', color='#2ca02c', lw=2, ms=6,
        label=f'Oracle (target==drafter)  mean={np.mean(oracle_acc):.2f}')
ax.axhline(0.88, color='#1f77b4', ls='--', lw=1.5,
           label='Trained EAGLE typical (0.85-0.92)')
ax.set_xlabel('Prompt index', fontsize=11)
ax.set_ylabel('Acceptance rate', fontsize=11)
ax.set_title(f'EAGLE Acceptance Rate (gamma={GAMMA})', fontsize=12)
ax.set_ylim(-0.05, 1.1)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 2: theoretical speedup surface
ax = fig.add_subplot(gs[0, 1])
alphas = np.linspace(0.5, 1.0, 100)
for g, col in [(2,'#d62728'),(4,'#ff7f0e'),(6,'#2ca02c'),(8,'#1f77b4')]:
    ax.plot(alphas * 100, 1 + g * alphas, lw=2, color=col, label=f'gamma={g}')
ax.axvline(70, color='gray', ls=':', alpha=0.6)
ax.axvline(90, color='gray', ls=':', alpha=0.6)
ax.text(70.5, 1.2, 'Indep.\nmodel', fontsize=8, color='gray')
ax.text(90.5, 1.2, 'EAGLE\ntrained', fontsize=8, color='gray')
ax.set_xlabel('Per-token acceptance rate (%)', fontsize=11)
ax.set_ylabel('Tokens generated per target call', fontsize=11)
ax.set_title('Theoretical Speedup vs Acceptance Rate', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 3: method comparison bar chart
ax = fig.add_subplot(gs[1, 0])
methods = ['Autoregressive\n(baseline)', 'Spec. decoding\n(small model)',
           'EAGLE\n(trained)', 'EAGLE-2\n(dynamic gamma)']
speedups = [1.0, 2.1, 3.0, 3.4]
colors   = ['#aec7e8', '#d62728', '#2ca02c', '#1f77b4']
bars = ax.bar(methods, speedups, color=colors, alpha=0.85, width=0.5)
for b, v in zip(bars, speedups):
    ax.text(b.get_x() + b.get_width()/2, v + 0.03, f'{v}x',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Throughput speedup (representative)', fontsize=11)
ax.set_title('Speedup Comparison on 7B Model\n(Representative published values)', fontsize=12)
ax.set_ylim(0, 4.2)
ax.grid(alpha=0.3, axis='y')

# Panel 4: EAGLE-2 dynamic draft length
ax = fig.add_subplot(gs[1, 1])
taus       = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
mean_gamma = [5.8, 5.2, 4.5, 3.6, 2.9, 2.1]   # simulated: stops earlier at higher tau
eff_speed  = [1 + g * 0.88 for g in mean_gamma] # 0.88 = typical trained acceptance
ax.bar(taus, eff_speed, width=0.07, color='#2ca02c', alpha=0.85)
ax.axhline(1 + 5 * 0.88, color='#d62728', ls='--', lw=2, label='Fixed gamma=5')
ax.set_xlabel('Confidence threshold tau', fontsize=11)
ax.set_ylabel('Effective speedup (1 + gamma * alpha)', fontsize=11)
ax.set_title('EAGLE-2 Dynamic Gamma\nStops drafting early when drafter is uncertain', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis='y')

plt.suptitle('EAGLE Decoding Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/10_eagle.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean acceptance -- random drafter: {np.mean(random_acc):.2f}')
print(f'Mean acceptance -- oracle:         {np.mean(oracle_acc):.2f}')
print()
print('Published EAGLE results on LLaMA-2-7B (trained drafter, MT-Bench):')
print('  Acceptance rate: 0.88-0.92')
print('  Speedup vs autoregressive: 2.5-3.5x')


## 4. Engineering Notes

**Training the EAGLE drafter**

```
1. Collect 100 K (prompt, continuation) pairs from the target model domain.
2. Run the target model on each continuation, saving per-step hidden states.
3. Train the drafter to minimise:
     L_total = L_ce(token prediction) + lambda * L_mse(hidden state prediction)
4. Training converges in 1-3 hours on a single A100 with a 7B target.
5. Save the drafter as a standalone checkpoint (~150 MB for a 7B target).
```

**Comparison: EAGLE vs other speculative methods**

| Method | Acceptance rate | Training cost | Memory overhead |
|--------|----------------|---------------|----------------|
| Small model (independent) | 65-78% | Pretrain from scratch | Full small model |
| EAGLE (trained) | 85-92% | Fine-tune 1 layer, ~2 h | ~150 MB |
| EAGLE-2 (dynamic gamma) | 85-92% + 10-20% more tokens | Same as EAGLE | Same as EAGLE |
| Medusa (see 10.03) | 75-85% | Fine-tune multiple heads, ~1 h | ~50-100 MB |

## 5. Exercises

1. **Training loss**: Implement the two-part EAGLE training loss `L = L_ce + 0.1 * L_mse`. Given a batch of hidden states collected from GPT-2, run 50 gradient steps and plot both loss components. Do they decrease together?

2. **Drafter depth ablation**: Implement EAGLE with 1, 2, and 3 decoder layers. After 100 forward passes, compare how often each depth produces the same top-1 token as the target model. Is the improvement from depth 1 to 2 significant?

3. **EAGLE-2 calibration**: Run 50 prompts with EAGLE-2 at tau values [0.1, 0.2, 0.3, 0.4, 0.5]. For each tau, record mean gamma used and mean acceptance rate. Plot gamma * acceptance_rate vs tau. What is the optimal tau?

4. **Feature similarity**: After running GPT-2 on 20 prompts, compute cosine similarity between consecutive hidden states f_t and f_{t+1}. Does high similarity correlate with higher acceptance rate on the next step?

5. **Wall-clock comparison**: Implement standard speculative decoding (GPT-2 small drafts for GPT-2 medium target) and EAGLE on the same target. Measure seconds per 100 output tokens on 5 prompts. Which method gives higher throughput on your hardware?

---

## 6. References

- [Li et al. (2024) -- EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty](https://arxiv.org/abs/2401.15077)
- [Li et al. (2024) -- EAGLE-2: Faster Inference of Language Models with Dynamic Draft Trees](https://arxiv.org/abs/2406.16858)
- [Chen et al. (2023) -- Accelerating Large Language Model Decoding with Speculative Sampling](https://arxiv.org/abs/2302.01318)
